# 02 — Featurisation Raman (classique + TFUG)

Objectif: dériver un vecteur de features robuste à partir d'un spectre prétraité.

Inclus:
- Détection de pics (argrelmax simple, fenêtres adaptatives)
- Descripteurs de forme (amplitude, aire, FWHM approximée, asymétrie locale)
- Ratios physiquement informés (ex.: paires de pics)
- **FWT** (ondelettes fractales simplifiées: énergie multi-échelles sur binaires fixes)
- **Dimension fractale D_f** (approx. box-counting/Higuchi-like sur le profil)
- **TFUQ** (moments « quaternioniques » simulés par couplages quadratiques entre bandes)
- Agrégation → DataFrame + export Parquet

## 0) Chargement d'un spectre prétraité
On s'attend à trouver `data/processed/<name>_preprocessed.parquet`. À défaut, on génère un spectre synthétique propre.

In [ ]:

import os, numpy as np, pandas as pd, glob

processed_dir = os.path.join("..","data","processed")
parqs = sorted(glob.glob(os.path.join(processed_dir, "*_preprocessed.parquet")))
if parqs:
    path = parqs[0]
    df = pd.read_parquet(path)
    wn = df["wavenumber"].values.astype(float)
    y  = df["intensity"].values.astype(float)
else:
    # spectre synthétique: 2 pics gaussiens + bruit
    rng = np.random.default_rng(0)
    wn = np.linspace(100, 1800, 1200)
    def g(mu, sigma, amp):
        return amp*np.exp(-0.5*((wn-mu)/sigma)**2)
    y = g(520, 6, 1.0) + g(1000, 12, 0.6) + 0.02*rng.normal(size=wn.size)
    y = (y - y.min())/(y.max()-y.min())
wn[:5], y[:5]


## 1) Détection de pics
Approche simple et robuste: on calcule un score local (diff filtrée) et on retient les maxima au-dessus d'un seuil adaptatif.

In [ ]:

import numpy as np

def detect_peaks(wn, y, window=9, min_prom=0.02):
    y = np.asarray(y, float)
    if window % 2 == 0: window += 1
    half = window//2
    # dérivée lissée (fenêtre moyenne)
    y_s = np.copy(y)
    y_s[half:-half] = np.convolve(y, np.ones(window)/window, mode="valid")
    # maxima locaux: y[i] > voisins
    cand = []
    for i in range(1, len(y)-1):
        if y_s[i] > y_s[i-1] and y_s[i] > y_s[i+1]:
            cand.append(i)
    cand = np.array(cand, int)
    if cand.size == 0:
        return np.array([], int)
    # proéminence approximative
    thr = min_prom * (np.max(y) - np.min(y) + 1e-12)
    peaks = [i for i in cand if (y[i] - min(y[max(0,i-half):i+half+1])) >= thr]
    return np.array(peaks, int)

peaks = detect_peaks(wn, y, window=11, min_prom=0.02)
peaks[:10], wn[peaks[:5]]


## 2) Descripteurs par pic (amplitude, aire, FWHM ~, asymétrie)

In [ ]:

def local_area(x, y, i, w=20):
    lo, hi = max(0, i-w), min(len(x)-1, i+w)
    return float(np.trapz(y[lo:hi+1], x[lo:hi+1]))

def local_fwhm(x, y, i, w=30):
    # approx: largeur à mi-hauteur par interpolation linéaire
    yc = y[i]
    half = yc/2.0
    lo, hi = max(0, i-w), min(len(x)-1, i+w)
    # gauche
    iL = i
    while iL>lo and y[iL] > half:
        iL -= 1
    # droite
    iR = i
    while iR<hi and y[iR] > half:
        iR += 1
    return float(x[iR]-x[iL]) if iR>iL else float(0.0)

def local_asymmetry(x, y, i, w=20):
    lo, hi = max(0, i-w), min(len(x)-1, i+w)
    left  = y[i]-y[lo]
    right = y[i]-y[hi]
    denom = (abs(left)+abs(right)+1e-12)
    return float((right-left)/denom)

peak_rows = []
for i in peaks:
    peak_rows.append({
        "peak_wn": float(wn[i]),
        "amp": float(y[i]),
        "area": local_area(wn, y, i, w=24),
        "fwhm": local_fwhm(wn, y, i, w=36),
        "asym": local_asymmetry(wn, y, i, w=24),
    })
import pandas as pd
df_peaks = pd.DataFrame(peak_rows).sort_values("amp", ascending=False).reset_index(drop=True)
df_peaks.head()


## 3) Ratios physiquement informés
Exemples: rapports d'amplitude/aire entre 2–3 pics dominants.

In [ ]:

def topk_ratios(df_peaks, k=3):
    d = df_peaks.nlargest(k, "amp").reset_index(drop=True)
    feats = {}
    if len(d)>=2:
        feats["ratio_amp_1_2"] = float(d.loc[0,"amp"]/ (d.loc[1,"amp"]+1e-12))
        feats["ratio_area_1_2"] = float(d.loc[0,"area"]/ (d.loc[1,"area"]+1e-12))
    if len(d)>=3:
        feats["ratio_amp_1_3"] = float(d.loc[0,"amp"]/ (d.loc[2,"amp"]+1e-12))
        feats["ratio_area_1_3"] = float(d.loc[0,"area"]/ (d.loc[2,"area"]+1e-12))
    return feats

ratios = topk_ratios(df_peaks, k=3)
ratios


## 4) FWT simplifiée — énergie multi-échelles
On partitionne l'axe en fenêtres (échelles) et on calcule l'énergie par bloc. Cela simule un banc d'ondelettes fractales.

In [ ]:

def multi_scale_energy(y, scales=(8,16,32,64,128)):
    y = np.asarray(y, float)
    feats = {}
    for s in scales:
        n = len(y)//s
        if n == 0:
            feats[f"fwt_s{s}_mean"] = float(np.mean(y**2))
            continue
        blocks = y[:n*s].reshape(n, s)
        feats[f"fwt_s{s}_mean"] = float(np.mean(np.mean(blocks**2, axis=1)))
        feats[f"fwt_s{s}_std"]  = float(np.std(np.mean(blocks**2, axis=1)))
    return feats

fwt_feats = multi_scale_energy(y)
fwt_feats


## 5) Dimension fractale \(D_f\) — approximation simple
Estimation rapide par **box-counting** 1D sur le profil.

In [ ]:

def df_boxcount(y, n_scales=8):
    y = (y - np.min(y))/(np.max(y)-np.min(y)+1e-12)
    N = len(y)
    sizes = np.unique(np.logspace(0, np.log10(N/8), n_scales, dtype=int))
    sizes = sizes[sizes>1]
    xs, Ns = [], []
    for s in sizes:
        n_boxes = int(np.ceil(N/s))
        cnt = 0
        for i in range(n_boxes):
            seg = y[i*s:min((i+1)*s, N)]
            if seg.size and (np.max(seg)-np.min(seg)) > 1e-3:
                cnt += 1
        xs.append(np.log(1.0/s))
        Ns.append(np.log(cnt+1e-12))
    # pente ~ D_f
    if len(xs) < 2: 
        return 1.0
    a = np.polyfit(xs, Ns, 1)[0]
    return float(max(0.5, min(2.0, a)))

Df = df_boxcount(y, n_scales=10)
Df


## 6) TFUQ — moments quadratiques simulant des couplages quaternioniques
On regroupe l'axe en 4 bandes et on calcule les produits croisés entre bandes (M12, M13, ...).

In [ ]:

def tfuq_quadratic(wn, y, n_bands=4):
    y = np.asarray(y, float)
    N = len(y)
    bsz = N//n_bands
    bands = [y[i*bsz:(i+1)*bsz] for i in range(n_bands)]
    means = [float(np.mean(b)) for b in bands]
    feats = {}
    for i in range(n_bands):
        feats[f"tfuq_m{i+1}"] = means[i]
    # couplages
    c = 1
    for i in range(n_bands):
        for j in range(i+1, n_bands):
            feats[f"tfuq_c{i+1}{j+1}"] = float(means[i]*means[j])
            c += 1
    return feats

tfuq = tfuq_quadratic(wn, y, n_bands=4)
tfuq


## 7) Agrégation → vecteur de features + export

In [ ]:

feats = {}
feats.update(ratios)
feats.update(fwt_feats)
feats.update({"Df": Df})
feats.update(tfuq)

import pandas as pd, os, json
feat_row = pd.DataFrame([feats])
outp = os.path.join("..","data")
feat_row.to_parquet(os.path.join(outp, "features.parquet"))
feat_row.T
